# 07 — Двухэтапное обучение ансамбля

## Пайплайн

```
┌─────────────────────────────────────────────────────────────────┐
│  ЭТАП 1: PRETRAIN  (~200-300k примеров, все источники)          │
│  • focal loss + class weights                                    │
│  • lr = 2e-5, 3 эпохи                                           │
└─────────────────┬───────────────────────────────────────────────┘
                  │ аугментация редких классов (rut5 + backtranslation)
                  ↓
┌─────────────────────────────────────────────────────────────────┐
│  ЭТАП 2: FINE-TUNE  (чистый нативный RU корпус, ~5-10k)         │
│  • cross-entropy + label smoothing                               │
│  • lr = 5e-6, 3 эпохи                                           │
└─────────────────────────────────────────────────────────────────┘
```


In [ ]:
import sys, os, json, gc

PROJECT_ROOT = '/kaggle/input/sentiment-analysis' if os.path.exists('/kaggle') else os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle') else '../results'
os.makedirs(WORKING_DIR, exist_ok=True)

import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_from_disk, Dataset, DatasetDict
from sklearn.utils import resample

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 1. Конфигурация

In [ ]:
from src.data_loader import EKMAN_LABEL_NAMES, EKMAN_ID2LABEL

LABEL_NAMES = EKMAN_LABEL_NAMES  # ['anger','disgust','fear','joy','sadness','surprise','neutral']
NUM_LABELS  = len(LABEL_NAMES)

# ── Три модели для ансамбля ────────────────────────────────────────────────
MODELS = {
    'rubert': {
        'model_name': 'blanchefort/rubert-base-cased-sentiment',
        'stage1_dir': f'{WORKING_DIR}/models/rubert_s1',
        'stage2_dir': f'{WORKING_DIR}/models/rubert',
        's1_batch_size': 32, 's2_batch_size': 32,
        's1_gradient_accumulation_steps': 1, 's2_gradient_accumulation_steps': 1,
    },
    'xlmroberta': {
        'model_name': 'xlm-roberta-base',
        'stage1_dir': f'{WORKING_DIR}/models/xlmroberta_s1',
        'stage2_dir': f'{WORKING_DIR}/models/xlmroberta',
        's1_batch_size': 16, 's2_batch_size': 16,
        's1_gradient_accumulation_steps': 2, 's2_gradient_accumulation_steps': 2,
    },
    'rubert_tiny': {
        'model_name': 'cointegrated/rubert-tiny2',
        'stage1_dir': f'{WORKING_DIR}/models/rubert_tiny_s1',
        'stage2_dir': f'{WORKING_DIR}/models/rubert_tiny',
        's1_batch_size': 64, 's2_batch_size': 64,
        's1_gradient_accumulation_steps': 1, 's2_gradient_accumulation_steps': 1,
    },
}

# ── Гиперпараметры обучения ────────────────────────────────────────────────
S1_EPOCHS    = 3;  S1_LR = 2e-5;  S1_LOSS = 'focal';  S1_GAMMA = 2.0
S2_EPOCHS    = 3;  S2_LR = 5e-6;  S2_LOSS = 'ce';     S2_SMOOTHING = 0.05
MAX_LENGTH   = 128
FP16         = True
SEED         = 42

# ── Аугментация ────────────────────────────────────────────────────────────
USE_AUGMENTATION       = True   # False — пропустить аугментацию (быстрее, но хуже)
AUG_METHOD             = 'both' # 'paraphrase' | 'backtranslation' | 'both'
AUG_TARGET_S1          = 3_000  # минимальное кол-во примеров на класс в Stage-1
AUG_TARGET_S2          = 400    # минимальное кол-во примеров на класс в Stage-2
AUG_N_VARIANTS         = 3      # парафразов на предложение (Stage-1)
AUG_N_VARIANTS_S2      = 5      # парафразов на предложение (Stage-2, датасет маленький)
AUG_BATCH_SIZE         = 8

# ── Какие модели обучать ───────────────────────────────────────────────────
TRAIN_FLAGS = {
    'rubert':      True,
    'xlmroberta':  True,
    'rubert_tiny': True,
}

print('Конфигурация загружена')


## 2. Сборка Stage-1 датасета

Большой смешанный корпус: GoEmotions (raw 211k) + CEDR + Izard + BRIGHTER + Aniemore + Dusha + опционально RuReviews/RuSentiTweet.

In [ ]:
from src.data_loader import (
    load_ru_go_emotions, load_cedr, load_ru_izard_emotions,
    load_brighter_hf, load_dusha, load_aniemore_resd,
    load_rureviews, load_rusentitweet, merge_datasets,
)
from src.preprocessor import preprocess_batch

S1_DATA_PATH     = f'{WORKING_DIR}/stage1_data'
S1_AUG_DATA_PATH = f'{WORKING_DIR}/stage1_data_augmented'

if os.path.exists(S1_DATA_PATH):
    print('Загружаем готовый Stage-1 датасет с диска...')
    stage1_ds = load_from_disk(S1_DATA_PATH)
else:
    print('Собираем Stage-1 датасет из всех источников...')
    s1_sources = {}

    # ru_go_emotions raw (~211k) даёт ~4× больше fear и disgust чем simplified
    for name, loader, kwargs in [
        ('ru_go_emotions_raw', load_ru_go_emotions, {'config': 'raw'}),
        ('cedr',               load_cedr,            {}),
        ('ru_izard',           load_ru_izard_emotions, {}),
    ]:
        try:
            s1_sources[name] = loader(**kwargs)
        except Exception as e:
            print(f'  WARNING: {name} failed: {e}')
            if name == 'ru_go_emotions_raw':
                try:
                    s1_sources['ru_go_emotions'] = load_ru_go_emotions(config='simplified')
                    print('  Fallback: loaded simplified GoEmotions')
                except Exception as e2:
                    print(f'  WARNING: GoEmotions simplified also failed: {e2}')

    for name, loader in [
        ('brighter_hf',  load_brighter_hf),
        ('aniemore',     load_aniemore_resd),
        ('rureviews',    load_rureviews),
        ('rusentitweet', load_rusentitweet),
    ]:
        try:
            s1_sources[name] = loader()
        except Exception as e:
            print(f'  WARNING: {name} failed: {e}')

    # Dusha: anger/sadness/neutral/joy (~300k транскриптов речи)
    try:
        s1_sources['dusha'] = load_dusha(max_samples=40_000)
    except Exception as e:
        print(f'  WARNING: dusha failed: {e}')

    stage1_raw = merge_datasets(s1_sources, test_size=0.15, val_size=0.15, seed=SEED)

    # Балансировка: не более 15k на класс
    MAX_PER_CLASS = 15_000
    train_df = stage1_raw['train'].to_pandas()

    print('\nРаспределение ДО балансировки:')
    for lid in range(NUM_LABELS):
        cnt = (train_df['label'] == lid).sum()
        print(f'  {EKMAN_ID2LABEL[lid]:<12}: {cnt:>7,}')

    parts = []
    for lid in range(NUM_LABELS):
        sub = train_df[train_df['label'] == lid]
        if len(sub) == 0:
            print(f'  WARNING: нет примеров для {EKMAN_ID2LABEL[lid]}')
            continue
        if len(sub) > MAX_PER_CLASS:
            sub = resample(sub, n_samples=MAX_PER_CLASS, random_state=SEED, replace=False)
        parts.append(sub)
    train_df = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

    stage1_ds = DatasetDict({
        'train':      Dataset.from_pandas(train_df),
        'validation': stage1_raw['validation'],
        'test':       stage1_raw['test'],
    })
    stage1_ds = stage1_ds.map(
        lambda b: preprocess_batch(b, clean=True), batched=True, batch_size=1000
    )
    stage1_ds.save_to_disk(S1_DATA_PATH)
    print(f'\nStage-1 сохранён: {S1_DATA_PATH}')

print('\nStage-1 splits:')
for split in stage1_ds:
    print(f'  {split:12s}: {len(stage1_ds[split]):,}')

print('\nРаспределение в Stage-1 train:')
df = stage1_ds['train'].to_pandas()
total = len(df)
for lid in range(NUM_LABELS):
    cnt = (df['label'] == lid).sum()
    bar = '█' * int(cnt / total * 40)
    print(f'  {EKMAN_ID2LABEL[lid]:<12}: {cnt:>6,}  ({cnt/total*100:4.1f}%)  {bar}')


## 3. Аугментация Stage-1 (редкие классы)

Парафраз через `cointegrated/rut5-base-paraphraser` + обратный перевод `Helsinki-NLP RU→EN→RU`.
Аугментируются только классы ниже порога `AUG_TARGET_S1`.

> Ориентировочное время на T4: ~10–20 мин при `method='both'`.  
> Если нужно быстрее — установи `AUG_METHOD = 'backtranslation'` (~5 мин) или `USE_AUGMENTATION = False`.


In [ ]:
from src.augmentation import augment_rare_classes

if not USE_AUGMENTATION:
    print('Аугментация отключена (USE_AUGMENTATION=False). Используем оригинальный Stage-1.')
    stage1_aug_ds = stage1_ds

elif os.path.exists(S1_AUG_DATA_PATH):
    print('Загружаем готовый аугментированный Stage-1 с диска...')
    stage1_aug_ds = load_from_disk(S1_AUG_DATA_PATH)

else:
    print(f'Аугментация Stage-1 (метод={AUG_METHOD}, цель={AUG_TARGET_S1}/класс)...')
    stage1_aug_ds = augment_rare_classes(
        dataset=stage1_ds,
        label_names=LABEL_NAMES,
        target_per_class=AUG_TARGET_S1,
        method=AUG_METHOD,
        n_variants=AUG_N_VARIANTS,
        batch_size=AUG_BATCH_SIZE,
        seed=SEED,
    )
    stage1_aug_ds.save_to_disk(S1_AUG_DATA_PATH)
    print(f'Stage-1 augmented сохранён: {S1_AUG_DATA_PATH}')

# Визуализация до/после
import matplotlib.pyplot as plt

df_before = stage1_ds['train'].to_pandas()
df_after  = stage1_aug_ds['train'].to_pandas()
c_before  = df_before['label'].value_counts().sort_index()
c_after   = df_after['label'].value_counts().sort_index()

COLORS = {'anger':'#e74c3c','disgust':'#8e44ad','fear':'#2c3e50',
          'joy':'#f39c12','sadness':'#3498db','surprise':'#1abc9c','neutral':'#95a5a6'}

if USE_AUGMENTATION:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, (counts, title) in zip(axes, [(c_before, 'До аугментации'), (c_after, 'После аугментации')]):
        labels = [EKMAN_ID2LABEL[i] for i in counts.index]
        ax.bar(labels, counts.values, color=[COLORS[l] for l in labels])
        ax.set_title(f'{title}\n(train: {counts.sum():,})')
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.savefig(f'{WORKING_DIR}/s1_augmentation.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nСравнение Stage-1:')
    print(f'  {"Класс":<12}  {"До":>8}  {"После":>8}  {"Δ":>7}')
    print('  ' + '-' * 44)
    for lid in range(NUM_LABELS):
        before = c_before.get(lid, 0)
        after  = c_after.get(lid, 0)
        delta  = after - before
        flag   = ' ← добавлено' if delta > 0 else ''
        print(f'  {LABEL_NAMES[lid]:<12}  {before:>8,}  {after:>8,}  +{delta:>5,}{flag}')
else:
    labels = [EKMAN_ID2LABEL[i] for i in c_before.index]
    plt.figure(figsize=(9, 3))
    plt.bar(labels, c_before.values, color=[COLORS[l] for l in labels])
    plt.title(f'Stage-1 train: {c_before.sum():,} примеров')
    plt.tight_layout(); plt.show()


## 4. Сборка Stage-2 датасета

Чистый нативный RU корпус: CEDR + BRIGHTER-HF + Aniemore/resd.

In [ ]:
from src.data_loader import load_stage2_clean

S2_DATA_PATH     = f'{WORKING_DIR}/stage2_data'
S2_AUG_DATA_PATH = f'{WORKING_DIR}/stage2_data_augmented'

if os.path.exists(S2_DATA_PATH):
    print('Загружаем готовый Stage-2 датасет с диска...')
    stage2_ds = load_from_disk(S2_DATA_PATH)
else:
    print('Собираем Stage-2 (чистый) датасет...')
    stage2_ds = load_stage2_clean(use_cedr=True, use_brighter_hf=True, use_aniemore=True, seed=SEED)
    stage2_ds = stage2_ds.map(
        lambda b: preprocess_batch(b, clean=True), batched=True, batch_size=500
    )
    stage2_ds.save_to_disk(S2_DATA_PATH)
    print(f'Stage-2 сохранён: {S2_DATA_PATH}')

print('\nStage-2 splits:')
for split in stage2_ds:
    print(f'  {split:12s}: {len(stage2_ds[split]):,}')

print('\nПокрытие классов в Stage-2 train:')
df2 = stage2_ds['train'].to_pandas()
total2 = len(df2)
for lid in range(NUM_LABELS):
    cnt = (df2['label'] == lid).sum()
    print(f'  {EKMAN_ID2LABEL[lid]:<12}: {cnt:>5,}  ({cnt/total2*100:.1f}%)')


## 5. Аугментация Stage-2 (редкие классы)

Stage-2 намеренно небольшой (~5-10k) — аугментируем только самые дефицитные классы до `AUG_TARGET_S2` примеров.


In [ ]:
if not USE_AUGMENTATION:
    print('Аугментация отключена. Используем оригинальный Stage-2.')
    stage2_aug_ds = stage2_ds

elif os.path.exists(S2_AUG_DATA_PATH):
    print('Загружаем готовый аугментированный Stage-2 с диска...')
    stage2_aug_ds = load_from_disk(S2_AUG_DATA_PATH)

else:
    print(f'Аугментация Stage-2 (метод={AUG_METHOD}, цель={AUG_TARGET_S2}/класс)...')
    stage2_aug_ds = augment_rare_classes(
        dataset=stage2_ds,
        label_names=LABEL_NAMES,
        target_per_class=AUG_TARGET_S2,
        method=AUG_METHOD,
        n_variants=AUG_N_VARIANTS_S2,
        batch_size=AUG_BATCH_SIZE,
        seed=SEED,
    )
    stage2_aug_ds.save_to_disk(S2_AUG_DATA_PATH)
    print(f'Stage-2 augmented сохранён: {S2_AUG_DATA_PATH}')

print('\nStage-2 (финальный для обучения):')
df2a = stage2_aug_ds['train'].to_pandas()
total2a = len(df2a)
for lid in range(NUM_LABELS):
    cnt = (df2a['label'] == lid).sum()
    flag = '' if cnt >= AUG_TARGET_S2 else '  ← мало данных!'
    print(f'  {EKMAN_ID2LABEL[lid]:<12}: {cnt:>5,}  ({cnt/total2a*100:.1f}%){flag}')


## 6. Двухэтапное обучение ансамбля

In [ ]:
from src.trainer import train_two_stage

# Используем аугментированные датасеты (или оригинальные если USE_AUGMENTATION=False)
_stage1 = stage1_aug_ds
_stage2 = stage2_aug_ds

all_results = {}

for model_key, cfg in MODELS.items():
    if not TRAIN_FLAGS.get(model_key, False):
        print(f'\nПропускаем {model_key}')
        continue

    print(f'\n{"#"*60}')
    print(f'  Модель: {cfg["model_name"]}')
    print(f'  Stage1: {cfg["stage1_dir"]}')
    print(f'  Stage2: {cfg["stage2_dir"]}')
    print('#'*60)

    os.makedirs(cfg['stage1_dir'], exist_ok=True)
    os.makedirs(cfg['stage2_dir'], exist_ok=True)

    model, tokenizer, r1, r2 = train_two_stage(
        model_name=cfg['model_name'],
        stage1_dataset=_stage1,
        stage2_dataset=_stage2,
        stage1_dir=cfg['stage1_dir'],
        stage2_dir=cfg['stage2_dir'],
        num_labels=NUM_LABELS,
        label_names=LABEL_NAMES,
        s1_epochs=S1_EPOCHS, s1_batch_size=cfg['s1_batch_size'], s1_lr=S1_LR,
        s1_loss_type=S1_LOSS, s1_focal_gamma=S1_GAMMA, s1_use_class_weights=True,
        s1_gradient_accumulation_steps=cfg['s1_gradient_accumulation_steps'],
        s2_epochs=S2_EPOCHS, s2_batch_size=cfg['s2_batch_size'], s2_lr=S2_LR,
        s2_loss_type=S2_LOSS, s2_label_smoothing=S2_SMOOTHING,
        s2_gradient_accumulation_steps=cfg['s2_gradient_accumulation_steps'],
        max_length=MAX_LENGTH, fp16=FP16, seed=SEED,
    )

    all_results[model_key] = {'stage1': r1, 'stage2': r2}
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('\n✓ Все модели обучены.')


## 7. Сравнение результатов

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.evaluation import evaluate_predictions, compare_models

rows = []
for model_key, cfg in MODELS.items():
    for stage, stage_dir in [('Stage 1', cfg['stage1_dir']), ('Stage 2 (final)', cfg['stage2_dir'])]:
        preds_path = os.path.join(stage_dir, 'test_preds.npy')
        if not os.path.exists(preds_path):
            continue
        preds  = np.load(os.path.join(stage_dir, 'test_preds.npy'))
        labels = np.load(os.path.join(stage_dir, 'test_labels.npy'))
        probs  = np.load(os.path.join(stage_dir, 'test_probs.npy'))
        m = evaluate_predictions(labels, preds, probs,
                                 model_name=f'{model_key} — {stage}',
                                 label_names=LABEL_NAMES)
        rows.append(m)

if rows:
    df = pd.DataFrame(rows).set_index('model')
    print('=== Stage 1 vs Stage 2 ===')
    print(df[['accuracy', 'f1_macro', 'f1_weighted']].round(4).to_string())

    compare_models(rows, save_path=f'{WORKING_DIR}/two_stage_comparison.png')

In [ ]:
# Per-class F1 после Stage 2 для каждой модели
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
if not axes.shape:
    axes = [axes]

for ax, (model_key, cfg) in zip(axes, MODELS.items()):
    stage2_dir = cfg['stage2_dir']
    if not os.path.exists(os.path.join(stage2_dir, 'test_preds.npy')):
        ax.set_title(f'{model_key}\n(не обучена)')
        continue

    preds  = np.load(os.path.join(stage2_dir, 'test_preds.npy'))
    labels = np.load(os.path.join(stage2_dir, 'test_labels.npy'))
    probs  = np.load(os.path.join(stage2_dir, 'test_probs.npy'))
    m = evaluate_predictions(labels, preds, probs, model_name=model_key, label_names=LABEL_NAMES)

    f1_per_class = [m.get(f'f1_{e}', 0) for e in LABEL_NAMES]
    colors = ['#e74c3c','#8e44ad','#2c3e50','#f39c12','#3498db','#1abc9c','#95a5a6']
    ax.barh(LABEL_NAMES, f1_per_class, color=colors)
    ax.set_xlim(0, 1)
    ax.set_title(f'{model_key}\nF1-macro = {m["f1_macro"]:.4f}')
    for i, v in enumerate(f1_per_class):
        ax.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=9)

axes[0].set_xlabel('F1-score')
plt.suptitle('Per-class F1 после двухэтапного обучения', fontsize=13)
plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/per_class_f1_two_stage.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Сохранение конфигурации для ансамблирования

In [ ]:
ensemble_config = {
    'model_dirs': {
        k: v['stage2_dir'] for k, v in MODELS.items()
        if os.path.exists(os.path.join(v['stage2_dir'], 'config.json'))
    },
    'label_names': LABEL_NAMES,
    'training_strategy': 'two_stage',
    'stage1': {
        'epochs': S1_EPOCHS, 'lr': S1_LR,
        'loss': S1_LOSS, 'gamma': S1_GAMMA,
    },
    'stage2': {
        'epochs': S2_EPOCHS, 'lr': S2_LR,
        'loss': S2_LOSS, 'label_smoothing': S2_SMOOTHING,
        'corpus': 'cedr + brighter_hf + aniemore/resd',
    },
}

with open(f'{WORKING_DIR}/ensemble_config.json', 'w', encoding='utf-8') as f:
    json.dump(ensemble_config, f, ensure_ascii=False, indent=2)

with open(f'{WORKING_DIR}/label_names.json', 'w') as f:
    json.dump(LABEL_NAMES, f)

print('Конфигурация сохранена:')
print(json.dumps(ensemble_config, ensure_ascii=False, indent=2))

---
## Дальнейшие шаги

1. **Ноутбук 05** — ансамблирование трёх финальных моделей (soft voting, stacking, temperature scaling)
2. **app/app.py** — Gradio-демо для интерактивного тестирования
3. Если F1 по disgust < 0.3 — попробуй увеличить `AUG_TARGET_S1` до 4000 и перезапустить
